# Chapter 4: File Layout and Partitioning

This notebook demonstrates the three-zone lake architecture and partitioning strategies.

## Setup

```bash
pip install duckdb polars pyarrow jupyter
```

## 1. Date Partitioning - Write Partitioned Parquet

Pattern 1: Most common use case (90% of scenarios)

In [1]:
import polars as pl
import os

# Create directory structure
os.makedirs("data/staging/orders", exist_ok=True)

# Generate sample data
from datetime import datetime, timedelta
import random

# Create 100K sample orders across 6 months
dates = [datetime(2024, 1, 1) + timedelta(days=i) for i in range(180)]
df = pl.DataFrame({
    "order_id": range(1, 100_001),
    "order_date": [random.choice(dates) for _ in range(100_000)],
    "revenue": [round(random.uniform(10, 500), 2) for _ in range(100_000)],
    "customer_id": [random.randint(1, 10000) for _ in range(100_000)],
})

# Save to staging first
df.write_parquet("data/staging/orders/snapshot_2024-01-15.parquet")
print(f"✓ Created staging file with {len(df):,} rows")

✓ Created staging file with 100,000 rows


In [2]:
# Read from staging and write partitioned to curated
df = pl.read_parquet("data/staging/orders/snapshot_2024-01-15.parquet")

# Add partition columns
df = df.with_columns([
    pl.col("order_date").dt.year().alias("year"),
    pl.col("order_date").dt.month().alias("month")
])

# Write partitioned
df.write_parquet(
    "data/curated/orders_fact/",
    partition_by=["year", "month"],
    mkdir=True
)

print("✓ Created partitioned structure in data/curated/orders_fact/")

✓ Created partitioned structure in data/curated/orders_fact/


## 2. Query Partitioned Data with DuckDB

DuckDB automatically prunes partitions based on filters

In [3]:
import duckdb

# Query only January 2024 (partition pruning)
result = duckdb.execute("""
    SELECT SUM(revenue) as total_revenue
    FROM read_parquet('data/curated/orders_fact/**/*.parquet')
    WHERE year = 2024 AND month = 1
""").fetchdf()

print("January 2024 Revenue:")
print(result)

January 2024 Revenue:
   total_revenue
0     4427567.29


## 3. Categorical Partitioning

Pattern 2: Partition by date AND category

In [4]:
# Add product category to our dataset
categories = ["electronics", "clothing", "food", "home"]

df = pl.read_parquet("data/staging/orders/snapshot_2024-01-15.parquet")
df = df.with_columns([
    pl.col("order_date").dt.year().alias("year"),
    pl.col("order_date").dt.month().alias("month"),
    pl.Series([random.choice(categories) for _ in range(len(df))]).alias("category")
])

# Partition by date AND category
df.write_parquet(
    "data/curated/orders_by_category/",
    partition_by=["year", "month", "category"],
)

print("✓ Created partitioned structure with category dimension")

✓ Created partitioned structure with category dimension


## 4. Hash-Based Bucketing

Pattern 3: For high-cardinality columns like customer_id

In [5]:
# Hash-based bucketing for customer events
df = pl.read_parquet("data/staging/orders/snapshot_2024-01-15.parquet")

df = df.with_columns([
    (pl.col("customer_id").hash() % 16).alias("bucket")
])

df.write_parquet(
    "data/curated/customer_events/",
    partition_by=["bucket"],
)

print("✓ Created 16 hash buckets for customer events")

✓ Created 16 hash buckets for customer events


## 5. No Partitioning for Small Tables

Pattern 4: Single file for small dimension tables

In [6]:
# Small product catalog - no partitioning needed
products_df = pl.DataFrame({
    "product_id": range(1, 1001),
    "product_name": [f"Product {i}" for i in range(1, 1001)],
    "category": [random.choice(categories) for _ in range(1000)],
    "price": [round(random.uniform(10, 500), 2) for _ in range(1000)]
})

os.makedirs("data/curated/product_catalog", exist_ok=True)
products_df.write_parquet("data/curated/product_catalog/products.parquet")

print(f"✓ Created single file with {len(products_df):,} products")

✓ Created single file with 1,000 products


## 6. Check Partition Sizes

In [7]:
import os
from pathlib import Path

def analyze_partitions(base_path):
    """Analyze partition sizes"""
    partitions = []
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith('.parquet'):
                path = os.path.join(root, file)
                size_mb = os.path.getsize(path) / 1e6
                partitions.append({
                    'path': path.replace(base_path + '/', ''),
                    'size_mb': round(size_mb, 2)
                })
    
    return pl.DataFrame(partitions).sort('size_mb', descending=True)

# Analyze orders_fact partitions
if os.path.exists('data/curated/orders_fact'):
    result = analyze_partitions('data/curated/orders_fact')
    print("Partition Sizes:")
    print(result)

Partition Sizes:
shape: (6, 2)
┌─────────────────────────────────┬─────────┐
│ path                            ┆ size_mb │
│ ---                             ┆ ---     │
│ str                             ┆ f64     │
╞═════════════════════════════════╪═════════╡
│ year=2024/month=1/00000000.par… ┆ 0.13    │
│ year=2024/month=3/00000000.par… ┆ 0.13    │
│ year=2024/month=4/00000000.par… ┆ 0.13    │
│ year=2024/month=5/00000000.par… ┆ 0.13    │
│ year=2024/month=6/00000000.par… ┆ 0.12    │
│ year=2024/month=2/00000000.par… ┆ 0.12    │
└─────────────────────────────────┴─────────┘


## 7. Example: Raw -> Staging -> Curated Pipeline

Complete ETL pipeline from Chapter 4

In [8]:
# Step 1: Simulate raw ingestion
from datetime import datetime, timezone
import gzip

os.makedirs("data/raw/shopify_orders", exist_ok=True)

# Generate raw CSV
raw_data = pl.DataFrame({
    "id": range(1, 10001),
    "email": [f"user{i}@example.com" for i in range(1, 10001)],
    "created_at": [datetime.now().strftime("%Y-%m-%d %H:%M:%S") for _ in range(10000)],
    "total_price": [round(random.uniform(10, 500), 2) for _ in range(10000)],
    "financial_status": [random.choice(["paid", "pending", "refunded"]) for _ in range(10000)],
    "fulfillment_status": [random.choice(["fulfilled", "pending", None]) for _ in range(10000)]
})

raw_file = f"data/raw/shopify_orders/{datetime.now(timezone.utc).isoformat()}.csv.gz"

# Write CSV with gzip compression
csv_content = raw_data.write_csv()
with gzip.open(raw_file, 'wt') as f:
    f.write(csv_content)

print(f"✓ Step 1: Raw data saved to {raw_file}")

✓ Step 1: Raw data saved to data/raw/shopify_orders/2026-05-25T10:45:58.813622+00:00.csv.gz


In [9]:
# Step 2: Transform to staging (clean + type)
df = pl.read_csv(raw_file)

staging_df = df.select([
    pl.col("id").cast(pl.Int64).alias("order_id"),
    pl.col("email").str.to_lowercase().alias("customer_email"),
    pl.col("created_at").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S").alias("order_date"),
    pl.col("total_price").cast(pl.Float64).alias("revenue"),
    pl.col("financial_status").alias("payment_status"),
    pl.col("fulfillment_status").alias("fulfillment_status")
]).filter(
    pl.col("revenue") > 0  # Drop test orders
)

staging_df.write_parquet("data/staging/orders/snapshot_pipeline.parquet")
print(f"✓ Step 2: Staged {staging_df.height:,} clean rows")

✓ Step 2: Staged 10,000 clean rows


In [10]:
# Step 3: Build curated (partition + optimize)
df = pl.read_parquet("data/staging/orders/snapshot_pipeline.parquet")

curated_df = df.with_columns([
    pl.col("order_date").dt.year().alias("year"),
    pl.col("order_date").dt.month().alias("month"),
    pl.col("order_date").dt.day().alias("day")
]).sort("order_date")  # Sort for better compression

curated_df.write_parquet(
    "data/curated/orders_pipeline/",
    partition_by=["year", "month"],
    compression="zstd",
    statistics=True
)

print(f"✓ Step 3: Curated {curated_df.height:,} optimized rows")

✓ Step 3: Curated 10,000 optimized rows


In [11]:
# Step 4: Query the curated layer
result = duckdb.execute("""
    SELECT
        order_date::DATE as day,
        COUNT(*) as orders,
        SUM(revenue) as total_revenue,
        AVG(revenue) as avg_order_value
    FROM read_parquet('data/curated/orders_pipeline/**/*.parquet', hive_partitioning=1)
    GROUP BY order_date::DATE
    ORDER BY day
    LIMIT 10
""").fetchdf()

print("\n✓ Step 4: Query results:")
print(result)


✓ Step 4: Query results:
         day  orders  total_revenue  avg_order_value
0 2026-05-25   10000     2549118.98       254.911898


## 8. Directory Structure Visualization

In [12]:
import os
from pathlib import Path

def print_tree(directory, prefix="", max_depth=3, current_depth=0):
    """Print directory tree structure"""
    if current_depth >= max_depth:
        return
    
    try:
        entries = sorted(Path(directory).iterdir(), key=lambda x: (not x.is_dir(), x.name))
        for i, entry in enumerate(entries):
            is_last = i == len(entries) - 1
            current_prefix = "└── " if is_last else "├── "
            print(f"{prefix}{current_prefix}{entry.name}")
            
            if entry.is_dir():
                extension_prefix = "    " if is_last else "│   "
                print_tree(entry, prefix + extension_prefix, max_depth, current_depth + 1)
    except PermissionError:
        pass

if os.path.exists('data'):
    print("Data Lake Structure:")
    print("data/")
    print_tree('data')

Data Lake Structure:
data/
├── curated
│   ├── customer_events
│   │   ├── bucket=0
│   │   ├── bucket=1
│   │   ├── bucket=10
│   │   ├── bucket=11
│   │   ├── bucket=12
│   │   ├── bucket=13
│   │   ├── bucket=14
│   │   ├── bucket=15
│   │   ├── bucket=2
│   │   ├── bucket=3
│   │   ├── bucket=4
│   │   ├── bucket=5
│   │   ├── bucket=6
│   │   ├── bucket=7
│   │   ├── bucket=8
│   │   └── bucket=9
│   ├── orders_by_category
│   │   └── year=2024
│   ├── orders_fact
│   │   └── year=2024
│   ├── orders_pipeline
│   │   └── year=2026
│   └── product_catalog
│       └── products.parquet
├── raw
│   └── shopify_orders
│       ├── 2026-05-25T10:45:43.115997+00:00.csv.gz
│       └── 2026-05-25T10:45:58.813622+00:00.csv.gz
└── staging
    └── orders
        ├── snapshot_2024-01-15.parquet
        └── snapshot_pipeline.parquet


## Summary

This notebook demonstrated:

1. **Three-zone architecture**: raw → staging → curated
2. **Date partitioning**: Most common pattern (year/month)
3. **Categorical partitioning**: Add dimension partitions
4. **Hash bucketing**: For high-cardinality columns
5. **No partitioning**: For small tables (<10M rows)
6. **Complete ETL pipeline**: End-to-end example

### Key Takeaways:
- Target 100-500MB per partition file
- Use Hive-style naming (`year=2024/month=01/`)
- DuckDB automatically prunes partitions
- Keep raw data immutable for disaster recovery